# CS781 Phase 5 — Experiment 1 (v5 HELDOUT + REFIT + ENSEMBLE)
## Model4 Partial / Layer-wise Fine-tuning + Statute-aware Hierarchical Chunks

**Task:** Legal Statute Identification (LSI), 100-label multilabel classification.

**Phase-1 starting point:** exact Efficient Modern Causal Decoder from the supplied Model4 Phase-1 notebook:
17 blocks, d=640, 10Q/2KV GQA, RoPE, RMSNorm, SwiGLU, 24,576-token LegalBPE, tied LM embeddings.

**Experiment-1 adaptation method:** Partial / layer-wise fine-tuning.
- freeze token embedding and lower 12 decoder blocks;
- train blocks 12–16, final RMSNorm, and the new task head;
- keep the backbone causal (do **not** convert it to a bidirectional encoder).

**Long-document representation:** four independent 384-token views:
1. HEAD
2. MIDDLE
3. STATUTE-EVIDENCE (densest local region containing statute cues)
4. TAIL

Each view uses `last-valid + masked-mean` pooling. A learned attention pool and max pool combine
the four views before a 100-logit multilabel classifier.

**Loss:** softened class-balanced BCE:
`sqrt((N-pos)/pos)` clipped to `[1, 10]`.

**Selection:** clean DEV Macro-F1 is primary. The notebook also reports official DEV
Macro-F1, Micro-F1 and Subset Accuracy. Threshold strategies include global,
per-label, and shrunk per-label thresholds.

This notebook intentionally predicts overlapping TEST texts with the model normally; it never copies
labels from TRAIN/DEV into TEST.

---
### v2 changes (core method untouched: same Model4, same 12-frozen/5-trainable split, same 4 statute-aware views, same attention+max pooling head, same loss)
1. **Threshold tuning rewritten** (vectorised, 0.01 grid, smoothed per-label F1 curves) and the strategy / checkpoint is now chosen by **2-fold cross-fitted DEV Macro-F1** instead of in-sample F1, so thresholds that over-fit DEV are no longer rewarded.
2. **Never-empty predictions**: if no label clears its threshold, the label with the highest prob/threshold ratio is emitted (every doc has >=1 gold label).
3. **EMA of trainable weights** (decay 0.999) used for evaluation/selection/test inference — typically a free +F1 and smoother checkpoints.
4. **3 epochs** (was 2), higher backbone/head LR, micro-batch 4 x accum 4 (same effective batch 16, better GPU use).
5. **Faster**: padding mask removed from attention (mathematically identical for right-padded inputs because the model is causal), and DEV is scored **once** (clean DEV is a subset of official DEV) instead of twice.
6. Saves **DEV and TEST probability matrices** (`.npz`) so this experiment can be ensembled with Exp 2-6.

---
### v3 review decisions
Kept from v2 because they are real improvements:
- EMA evaluation/checkpointing;
- cross-fitted threshold-strategy selection;
- finer threshold grid with smoothed per-label curves;
- never-empty multilabel predictions;
- one official-DEV inference pass with clean-DEV slicing;
- DEV/TEST probability matrices for later ensembling.

Safety changes in v3:
- restore conservative T4 micro-batch and learning rates from v1;
- keep 3 epochs, but checkpoint selection remains DEV-driven, so later training cannot overwrite a better earlier model;
- retain the explicit right-padding key mask by default (correctness first; fast no-mask mode remains optional);
- use the never-empty rule consistently inside cross-validation threshold scoring too;
- add an OOM-friendly note and versioned artifact names.

### v4 addition
The notebook now embeds the **exact Phase-5 evaluator script supplied by the course**, runs it as a real subprocess on official DEV predictions, and validates the actual `prediction.json` extracted from the final submission ZIP with the same evaluator parser.

### v5 additions — submission variants
This version keeps the heldout experiment unchanged and adds two extra TEST variants:

1. **Heldout** — the original best checkpoint selected on clean DEV.
2. **Refit** — a fresh Phase-1 Model4 initialization, trained on `TRAIN + official DEV`
   for the training duration selected by the heldout experiment.
3. **Ensemble** — simple 50/50 probability average of Heldout and Refit.

**Important calibration rule:** Refit has seen DEV labels, so DEV is *not* reused to tune its
thresholds or ensemble weight. Both Refit and Ensemble reuse the threshold vector selected by
the heldout experiment. This avoids post-refit DEV leakage in submission construction.

All three ZIPs contain exactly one `prediction.json` and are parsed with the exact supplied
`evaluate_phase5.py` prediction reader before being marked ready.


### Kaggle inputs

Attach:
- `eff_full_best*.pt` from Phase-1 Model4
- the exact Phase-1 `tokenizer.json`

Add your gated-dataset Hugging Face token in Kaggle Secrets under **`HF_TOKEN`**.

The notebook downloads only the official Phase-5 train/dev/test files.


In [1]:
# CELL 1 — imports, reproducibility, configuration
import os, re, gc, json, math, time, random, hashlib, zipfile, shutil, heapq, contextlib, subprocess, sys, importlib.util
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from sklearn.metrics import f1_score, accuracy_score
from huggingface_hub import hf_hub_download, login

SEED = 1234
NOTEBOOK_VERSION = "v5_HELDOUT_REFIT_ENSEMBLE"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

ROOT = Path("/kaggle/working/phase5_exp1_model4_partialft_v5")
CACHE_DIR = ROOT / "cache"
ROOT.mkdir(parents=True, exist_ok=True); CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Exact Phase-1 Model4 architecture constants.
VOCAB_SIZE = 24_576
D_MODEL = 640
N_LAYERS = 17
N_HEADS = 10
N_KV_HEADS = 2
FFN_DIM = 2048
DROPOUT = 0.0
MAX_POSITIONS = 2560
ROPE_BASE = 10_000.0
RMS_EPS = 1e-5

# Phase-5 task.
N_LABELS = 100
N_CHUNKS = 4
CHUNK_LEN = 384
FROZEN_BLOCKS = 12
TRAINABLE_TOP_BLOCKS = N_LAYERS - FROZEN_BLOCKS

# Training: conservative T4 defaults.
MICRO_BATCH_SIZE = 2          # safer on 16 GB T4; 4 chunks => 8 sequences/step
DEV_BATCH_SIZE = 8            # inference only; reduce to 4 if needed
GRAD_ACCUM_STEPS = 8          # effective batch = 16
EPOCHS = 3                    # best checkpoint can come from any eval point
EVALS_PER_EPOCH = 3
BACKBONE_LR = 4e-5            # conservative partial-FT LR
HEAD_LR = 4e-4                # conservative task-head LR
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
EMA_DECAY = 0.999             # v2: EMA of trainable weights (used for eval / selection / test)
FAST_CAUSAL_NO_MASK = False   # default: preserve explicit padding mask; True is optional speed mode
GRAD_CLIP = 1.0
USE_AMP = True
NUM_WORKERS = 2

# Threshold tuning.
THR_GRID = np.round(np.arange(0.02, 0.8001, 0.01), 3)   # v2: finer / wider grid (vectorised, cheap)
SHRINK_TAUS = [25.0, 50.0, 100.0, 200.0, 400.0]
CV_REPEATS = 2                                          # v2: repeated 2-fold cross-fit for honest threshold/strategy selection

# Optional final refit. Leave False for the first quick/controlled experiment.
RUN_REFIT = True
REFIT_SEED = 2234
ENSEMBLE_HELDOUT_WEIGHT = 0.50
ENSEMBLE_REFIT_WEIGHT = 0.50
REFIT_USE_SELECTED_WEIGHT_STYLE = True   # EMA if heldout winner was EMA; raw if raw won

# Optional asset overrides if auto-detection is ambiguous.
MODEL4_CKPT_OVERRIDE = None
TOKENIZER_OVERRIDE = None

def sha256_file(path, chunk=1<<20):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def norm_text(s):
    return re.sub(r"\s+"," ",str(s).lower()).strip()

def text_hash(s):
    return hashlib.sha256(norm_text(s).encode("utf-8")).hexdigest()

def atomic_torch_save(obj, path):
    path=Path(path); tmp=Path(str(path)+".tmp")
    torch.save(obj,tmp); os.replace(tmp,path)

print("ROOT:", ROOT)
print("Trainable top blocks:", TRAINABLE_TOP_BLOCKS)


Device: cuda
GPU: Tesla T4
ROOT: /kaggle/working/phase5_exp1_model4_partialft_v5
Trainable top blocks: 5


In [2]:
# CELL 2 — find Phase-1 Model4 checkpoint + exact tokenizer
all_files=[]
for base in ["/kaggle/input","/kaggle/working"]:
    if os.path.exists(base):
        for root,dirs,files in os.walk(base):
            for f in files:
                all_files.append(os.path.join(root,f))

ckpt_cands=[p for p in all_files if p.lower().endswith(".pt") and "eff_full_best" in os.path.basename(p).lower()]
tok_cands=[p for p in all_files if os.path.basename(p).lower()=="tokenizer.json"]

def asset_score(p, words):
    s=p.lower()
    return sum(10 for w in words if w in s) - len(p)/100000.0

if MODEL4_CKPT_OVERRIDE:
    MODEL4_CKPT=str(MODEL4_CKPT_OVERRIDE)
else:
    assert ckpt_cands, "Could not find eff_full_best*.pt. Attach the Phase-1 Model4 best checkpoint."
    MODEL4_CKPT=sorted(ckpt_cands,key=lambda p:asset_score(p,["model4","decoder-efficient","phase1","efficient"]),reverse=True)[0]

if TOKENIZER_OVERRIDE:
    TOKENIZER_FILE=str(TOKENIZER_OVERRIDE)
else:
    assert tok_cands, "Could not find tokenizer.json. Attach the exact Phase-1 Model4 tokenizer."
    # Prefer tokenizer close to the selected checkpoint / with Model4 hints.
    ckpt_dir=os.path.dirname(MODEL4_CKPT).lower()
    TOKENIZER_FILE=sorted(
        tok_cands,
        key=lambda p:(
            20 if os.path.dirname(p).lower()==ckpt_dir else 0,
            asset_score(p,["model4","decoder-efficient","phase1","complete-model"])
        ),
        reverse=True
    )[0]

print("Checkpoint candidates:")
for p in ckpt_cands: print("  ",p)
print("Tokenizer candidates:")
for p in tok_cands: print("  ",p)
print("\nSELECTED MODEL4:",MODEL4_CKPT)
print("SELECTED TOKENIZER:",TOKENIZER_FILE)
print("Model4 SHA256:",sha256_file(MODEL4_CKPT))
print("Tokenizer SHA256:",sha256_file(TOKENIZER_FILE))


Checkpoint candidates:
   /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
Tokenizer candidates:
   /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json

SELECTED MODEL4: /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
SELECTED TOKENIZER: /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json
Model4 SHA256: 529cfef4605822c82075aee8254259ea8a42b91ec76ca41e32bd99ea7b61ed74
Tokenizer SHA256: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [3]:
# CELL 3 — exact Phase-1 LegalBPE tokenizer class
# Cell 3 — Custom byte-level BPE (same assignment design, scaled up)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def fit(self, sentences, min_count=2):
        word_counts = Counter()
        seen_chars = 0
        t0 = time.time()

        for sentence in sentences:
            seen_chars += len(sentence)
            word_counts.update(re.findall(r"\S+|\s+", sentence))
            if seen_chars and seen_chars % 10_000_000 < len(sentence):
                print(f"  BPE corpus read: {seen_chars/1e6:.0f}M chars | unique units={len(word_counts):,}", flush=True)

        if min_count > 1:
            word_counts = Counter({w:c for w,c in word_counts.items() if c >= min_count})

        print(f"BPE training units after min_count={min_count}: {len(word_counts):,}")
        print(f"Counting time: {(time.time()-t0)/60:.1f} min")

        words = list(word_counts)
        freqs = [word_counts[w] for w in words]
        syms = [self.word_to_symbols(w) for w in words]

        target_merges = self.vocab_size - len(self.itos)
        pair_counts = Counter()
        pair_words = {}

        for wi, s in enumerate(syms):
            f = freqs[wi]
            for j in range(len(s)-1):
                p = (s[j], s[j+1])
                pair_counts[p] += f
                pair_words.setdefault(p, set()).add(wi)

        heap = [(-c,p) for p,c in pair_counts.items()]
        heapq.heapify(heap)

        for merge_num in range(target_merges):
            best_pair = None
            best_freq = 0
            while heap:
                negc, p = heap[0]
                if pair_counts.get(p,0) == -negc and -negc > 0:
                    best_pair, best_freq = p, -negc
                    break
                heapq.heappop(heap)

            if best_pair is None or best_freq <= 1:
                print("No useful pairs remain.")
                break

            left, right = best_pair
            new_symbol = left + right
            self.merges.append([left,right])
            self.symbol_bytes[new_symbol] = self.symbol_bytes[left] + self.symbol_bytes[right]
            if new_symbol not in self.stoi:
                self.stoi[new_symbol] = len(self.itos)
                self.itos.append(new_symbol)

            touched = set()
            for wi in list(pair_words.get(best_pair, ())):
                old = syms[wi]
                f = freqs[wi]

                # Remove old local pair contributions.
                for j in range(len(old)-1):
                    p = (old[j], old[j+1])
                    pair_counts[p] -= f
                    touched.add(p)

                merged = []
                j = 0
                while j < len(old):
                    if j+1 < len(old) and old[j] == left and old[j+1] == right:
                        merged.append(new_symbol)
                        j += 2
                    else:
                        merged.append(old[j])
                        j += 1
                syms[wi] = merged

                # Add new local pair contributions.
                for j in range(len(merged)-1):
                    p = (merged[j], merged[j+1])
                    pair_counts[p] += f
                    pair_words.setdefault(p, set()).add(wi)
                    touched.add(p)

            pair_words.pop(best_pair, None)

            for p in touched:
                c = pair_counts.get(p,0)
                if c > 0:
                    heapq.heappush(heap, (-c,p))
                else:
                    pair_counts.pop(p, None)

            if (merge_num+1) % 1000 == 0 or merge_num == 0 or merge_num+1 == target_merges:
                print(
                    f"merge {merge_num+1:,}/{target_merges:,} | "
                    f"freq={best_freq:,} | vocab={len(self.itos):,}",
                    flush=True
                )

        self.vocab_size = len(self.itos)
        self._reset_encode_cache()
        print(f"BPE complete: vocab={len(self):,}")
        return self

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        # Bound cache memory for 500M-token preprocessing.
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    def save_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {"vocab_size":len(self), "merges":self.merges, "itos":self.itos},
                f, ensure_ascii=False
            )

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s=f"{b:02x}"
            tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

print("Tokenizer class ready.")

tokenizer = LegalBPE.load_json(TOKENIZER_FILE)
print("Tokenizer vocab:",len(tokenizer))
print("special ids:",tokenizer.pad_token_id,tokenizer.unk_token_id,tokenizer.bos_token_id,tokenizer.eos_token_id)
assert len(tokenizer)==VOCAB_SIZE
assert tokenizer.pad_token_id==0
print("✅ tokenizer sanity PASS")


Tokenizer class ready.
Tokenizer vocab: 24576
special ids: 0 1 2 3
✅ tokenizer sanity PASS


In [4]:
# CELL 4 — authenticate and download official Phase-5 data
from kaggle_secrets import UserSecretsClient
HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
assert HF_TOKEN and len(HF_TOKEN)>10, "HF_TOKEN secret missing."
login(token=HF_TOKEN,add_to_git_credential=False)

HF_REPO="Exploration-Lab/CS-781-Capstone"
FNS={
    "train":"Phase_5_LSI/Train/train.json",
    "dev":"Phase_5_LSI/Train/dev.json",
    "test":"Phase_5_LSI/Test/test.json",
}
paths={k:hf_hub_download(repo_id=HF_REPO,repo_type="dataset",filename=v,token=HF_TOKEN) for k,v in FNS.items()}
DATA_SHA256={k:sha256_file(v) for k,v in paths.items()}
print("Dataset SHA256:")
for k,v in DATA_SHA256.items(): print(" ",k,v)

def read_json(path):
    with open(path,encoding="utf-8") as f: return json.load(f)

train_raw=read_json(paths["train"])
dev_raw=read_json(paths["dev"])
test_raw=read_json(paths["test"])
print("raw rows:",len(train_raw),len(dev_raw),len(test_raw))
assert len(train_raw)==42750 and len(dev_raw)==10181 and len(test_raw)==13019

for name,recs,labeled in [("TRAIN",train_raw,True),("DEV",dev_raw,True),("TEST",test_raw,False)]:
    ids=[str(r["id"]) for r in recs]
    assert len(ids)==len(set(ids)),f"{name} duplicate IDs"
    for r in recs:
        assert isinstance(r.get("text"),str)
        if labeled:
            assert isinstance(r.get("labels"),list)
            # Dataset contains a few duplicate label entries; collapse duplicates as set semantics.
            r["labels"]=sorted(set(int(x) for x in r["labels"]))
            assert all(0<=x<N_LABELS for x in r["labels"])

assert not any("labels" in r for r in test_raw),"TEST labels unexpectedly public."
print("✅ schema / ID / label sanity PASS")


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Dataset SHA256:
  train f3601a7fce994616daf99fec251ca7936e98675a8fbf8af20a782cd55054c8a7
  dev ab2660ac23abf38e902d63ab6871df740dc38e3fd973f441745f42564dadbd15
  test c699d50bcc2f3e369955268d6195a80b23322ed91cb1fadc8d58b26ca8e9aa77
raw rows: 42750 10181 13019
✅ schema / ID / label sanity PASS


In [5]:
# CELL 5 — leakage-aware split audit and clean DEV
train_hashes={text_hash(r["text"]) for r in train_raw}
dev_hashes={text_hash(r["text"]) for r in dev_raw}
test_hashes={text_hash(r["text"]) for r in test_raw}

print("unique normalized texts:",len(train_hashes),len(dev_hashes),len(test_hashes))
print("text overlap train-dev:",len(train_hashes & dev_hashes))
print("text overlap train-test:",len(train_hashes & test_hashes))
print("text overlap dev-test:",len(dev_hashes & test_hashes))

# Clean DEV excludes any DEV row whose normalized text occurs anywhere in TRAIN.
CLEAN_IDX=np.asarray([i for i,r in enumerate(dev_raw) if text_hash(r["text"]) not in train_hashes],dtype=np.int64)
clean_dev=[dev_raw[i] for i in CLEAN_IDX]   # v2: clean DEV is a subset of official DEV -> score official DEV once, slice
print("official DEV:",len(dev_raw),"clean DEV:",len(clean_dev),"removed rows:",len(dev_raw)-len(clean_dev))
assert len(clean_dev)>9000
assert not ({text_hash(r["text"]) for r in clean_dev} & train_hashes)

# Keep TRAIN rows even when normalized text repeats because the audit found conflicting label sets.
train_records=train_raw
test_records=test_raw

train_counts=np.zeros(N_LABELS,dtype=np.int64)
for r in train_records:
    train_counts[r["labels"]]+=1
dev_counts=np.zeros(N_LABELS,dtype=np.int64)
for r in clean_dev:
    dev_counts[r["labels"]]+=1

print("TRAIN label count min/median/max:",train_counts.min(),np.median(train_counts),train_counts.max())
print("clean DEV label count min/median/max:",dev_counts.min(),np.median(dev_counts),dev_counts.max())
assert np.all(train_counts>0)
assert np.all(dev_counts>0),"Clean DEV lost at least one label; threshold tuning would be unsafe."

raw_pw=(len(train_records)-train_counts)/train_counts
POS_WEIGHT=np.clip(np.sqrt(raw_pw),1.0,10.0).astype(np.float32)
print("softened pos_weight min/median/max:",POS_WEIGHT.min(),np.median(POS_WEIGHT),POS_WEIGHT.max())

# Save official and clean DEV truth for exact evaluator-compatible checks later.
official_truth=[{"id":str(r["id"]),"gold":[int(x) for x in r["labels"]]} for r in dev_raw]
clean_truth=[{"id":str(r["id"]),"gold":[int(x) for x in r["labels"]]} for r in clean_dev]
json.dump(official_truth,open(ROOT/"dev_truth_official.json","w"))
json.dump(clean_truth,open(ROOT/"dev_truth_clean.json","w"))


unique normalized texts: 42315 10118 12939
text overlap train-dev: 81
text overlap train-test: 109
text overlap dev-test: 62
official DEV: 10181 clean DEV: 10063 removed rows: 118
TRAIN label count min/median/max: 272 679.0 11730
clean DEV label count min/median/max: 66 173.0 2880
softened pos_weight min/median/max: 1.6261922 7.871644 10.0


In [6]:
# CELL 6 — fast statute-aware view selection + reusable uint16 token cache
# We preserve exact text within character windows; only the four selected regions are BPE-tokenized.

CUE_RE=re.compile(
    r"\b(section|sections|sec\.?|article|articles|act|acts|rule|rules|"
    r"provision|provisions|ipc|crpc|constitution)\b", re.I
)
CHAR_WINDOW=6000

def _char_window(text, center, width=CHAR_WINDOW):
    if len(text)<=width: return text
    st=max(0,min(len(text)-width,int(center-width//2)))
    return text[st:st+width]

def select_views(text):
    n=len(text)
    if n<=CHAR_WINDOW:
        head=mid=evid=tail=text
        return [head,mid,evid,tail]

    head=text[:CHAR_WINDOW]
    tail=text[-CHAR_WINDOW:]
    mid=_char_window(text,n//2)

    hits=[m.start() for m in CUE_RE.finditer(text)]
    if hits:
        # Densest CHAR_WINDOW region. Two-pointer O(number of cues).
        best_i=best_j=0; j=0
        for i in range(len(hits)):
            if j<i: j=i
            while j+1<len(hits) and hits[j+1]-hits[i]<=CHAR_WINDOW:
                j+=1
            if j-i>best_j-best_i:
                best_i,best_j=i,j
        center=(hits[best_i]+hits[best_j])//2
        evid=_char_window(text,center)
    else:
        evid=mid
    return [head,mid,evid,tail]

def encode_view(text, view_idx):
    ids=tokenizer.encode(text)
    if not ids: ids=[tokenizer.unk_token_id]
    if len(ids)>CHUNK_LEN:
        if view_idx==0: ids=ids[:CHUNK_LEN]            # head
        elif view_idx==3: ids=ids[-CHUNK_LEN:]         # tail
        else:
            st=(len(ids)-CHUNK_LEN)//2                 # middle/evidence keep center
            ids=ids[st:st+CHUNK_LEN]
    return ids

def cache_split(name, records):
    ids_path=CACHE_DIR/f"{name}_ids.npy"
    len_path=CACHE_DIR/f"{name}_lens.npy"
    meta_path=CACHE_DIR/f"{name}_meta.json"

    fingerprint={
        "dataset_sha256":DATA_SHA256["dev" if name.startswith("dev") else name],
        "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
        "n":len(records),"n_chunks":N_CHUNKS,"chunk_len":CHUNK_LEN,
        "char_window":CHAR_WINDOW,
        "ids_hash":hashlib.sha256("|".join(str(r["id"]) for r in records).encode()).hexdigest(),
    }

    if ids_path.exists() and len_path.exists() and meta_path.exists():
        old=json.load(open(meta_path))
        if old==fingerprint:
            print("✅ reuse token cache:",name)
            return np.load(ids_path,mmap_mode="r"),np.load(len_path,mmap_mode="r")
        print("cache fingerprint changed; rebuilding:",name)

    arr=np.full((len(records),N_CHUNKS,CHUNK_LEN),tokenizer.pad_token_id,dtype=np.uint16)
    lens=np.zeros((len(records),N_CHUNKS),dtype=np.uint16)
    t0=time.time()
    for i,r in enumerate(records):
        for c,txt in enumerate(select_views(r["text"])):
            z=encode_view(txt,c)
            L=min(len(z),CHUNK_LEN)
            arr[i,c,:L]=np.asarray(z[:L],dtype=np.uint16)
            lens[i,c]=L
        if (i+1)%2000==0 or i+1==len(records):
            print(f"{name}: {i+1}/{len(records)} elapsed={(time.time()-t0)/60:.1f}m",flush=True)

    np.save(ids_path,arr); np.save(len_path,lens)
    json.dump(fingerprint,open(meta_path,"w"),indent=2)
    del arr,lens; gc.collect()
    return np.load(ids_path,mmap_mode="r"),np.load(len_path,mmap_mode="r")

train_tok,train_lens=cache_split("train",train_records)
official_tok,official_lens=cache_split("dev",dev_raw)
test_tok,test_lens=cache_split("test",test_records)

print("cache shapes:",train_tok.shape,official_tok.shape,test_tok.shape)
assert train_tok.shape==(len(train_records),4,384)


train: 2000/42750 elapsed=0.1m
train: 4000/42750 elapsed=0.3m
train: 6000/42750 elapsed=0.4m
train: 8000/42750 elapsed=0.5m
train: 10000/42750 elapsed=0.6m
train: 12000/42750 elapsed=0.7m
train: 14000/42750 elapsed=0.9m
train: 16000/42750 elapsed=1.0m
train: 18000/42750 elapsed=1.1m
train: 20000/42750 elapsed=1.2m
train: 22000/42750 elapsed=1.3m
train: 24000/42750 elapsed=1.5m
train: 26000/42750 elapsed=1.6m
train: 28000/42750 elapsed=1.7m
train: 30000/42750 elapsed=1.8m
train: 32000/42750 elapsed=1.9m
train: 34000/42750 elapsed=2.1m
train: 36000/42750 elapsed=2.2m
train: 38000/42750 elapsed=2.3m
train: 40000/42750 elapsed=2.4m
train: 42000/42750 elapsed=2.5m
train: 42750/42750 elapsed=2.6m
dev: 2000/10181 elapsed=0.1m
dev: 4000/10181 elapsed=0.3m
dev: 6000/10181 elapsed=0.4m
dev: 8000/10181 elapsed=0.5m
dev: 10000/10181 elapsed=0.7m
dev: 10181/10181 elapsed=0.7m
test: 2000/13019 elapsed=0.1m
test: 4000/13019 elapsed=0.3m
test: 6000/13019 elapsed=0.4m
test: 8000/13019 elapsed=0.5m
test

In [7]:
# CELL 7 — exact Phase-1 Model4 classes (copied from supplied Phase-1 notebook)
# Cell 9 — ~99.31M Efficient Modern Decoder: GQA + RoPE + RMSNorm + SwiGLU   (v2)
# Same parameters / same state_dict keys / same maths as v1 (v1 checkpoints load unchanged). What changed:
#   * RoPE cos/sin are gathered ONCE per forward (v1: once per layer = 34 gathers + 17 GPU->CPU syncs per step)
#   * decode steps use the GQA-native path: the 5 query heads of a KV group become 5 query rows, so the cached
#     K/V are never expanded 5x (v1 copied the whole expanded cache every layer at every generated token)
#   * multi-token chunks that follow a cache are now causal (v1 silently made them bidirectional; unused so far)
#   * build_model() so training always starts from fresh weights; embeddings excluded from weight decay in Cell 11

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x32.to(dtype) * self.weight)


class RotaryTable(nn.Module):
    """cos/sin table (non-persistent buffers -> nothing is added to the state_dict)."""
    def __init__(self, head_dim, max_positions, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", freqs.cos(), persistent=False)
        self.register_buffer("sin_cached", freqs.sin(), persistent=False)
        self.max_positions = max_positions

    def forward(self, position_ids):
        # (B,T) -> two (B,1,T,D/2) tensors, shared by every layer
        return (self.cos_cached[position_ids].unsqueeze(1),
                self.sin_cached[position_ids].unsqueeze(1))


def apply_rope(x, cos, sin):
    # x: (B,H,T,D) with interleaved (even, odd) rotation pairs -- identical convention to v1
    B, H, T, D = x.shape
    cos = cos.to(x.dtype)
    sin = sin.to(x.dtype)
    xp = x.view(B, H, T, D // 2, 2)
    x0, x1 = xp[..., 0], xp[..., 1]
    return torch.stack((x0 * cos - x1 * sin, x0 * sin + x1 * cos), dim=-1).flatten(-2)


class GQACausalSelfAttention(nn.Module):
    GQA_DECODE_PATH = True     # flip to False to fall back to the (slower) expand-K/V path; used by the self-test

    def __init__(self, d_model, n_heads, n_kv_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.kv_repeat = n_heads // n_kv_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape
        H, KV, D = self.n_heads, self.n_kv_heads, self.head_dim

        q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)
        k = self.k_proj(x).view(B, T, KV, D).transpose(1, 2)
        v = self.v_proj(x).view(B, T, KV, D).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        new_kv = (k, v) if use_cache else None          # compact cache: only KV heads are stored
        S = k.size(2)
        drop = self.dropout if self.training else 0.0

        # ---------------- single new token (cached decoding): GQA-native, no K/V expansion
        if T == 1 and past_len > 0 and self.GQA_DECODE_PATH:
            qg = q.reshape(B, KV, self.kv_repeat, D)     # head h = kv*rep + r  ->  row r of group kv
            mask = None if key_valid is None else key_valid[:, None, None, :]
            y = F.scaled_dot_product_attention(qg, k, v, attn_mask=mask, dropout_p=drop, is_causal=False)
            y = y.reshape(B, H, 1, D).transpose(1, 2).contiguous().view(B, 1, C)
            return self.o_proj(y), new_kv

        # ---------------- general path (training / prefill / chunks)
        if self.kv_repeat != 1:
            k_attn = k.repeat_interleave(self.kv_repeat, dim=1)
            v_attn = v.repeat_interleave(self.kv_repeat, dim=1)
        else:
            k_attn, v_attn = k, v

        attn_mask, is_causal = None, False
        if T == 1:
            if key_valid is not None:
                attn_mask = key_valid[:, None, None, :]
        elif key_valid is None and past_len == 0:
            is_causal = True                             # fast path used by training
        else:
            qi = torch.arange(T, device=x.device)[:, None] + past_len
            kj = torch.arange(S, device=x.device)[None, :]
            attn_mask = (kj <= qi)[None, None, :, :]
            if key_valid is not None:
                attn_mask = attn_mask & key_valid[:, None, None, :]
                if past_len == 0:                        # left-padding queries keep one safe key (themselves)
                    attn_mask = attn_mask | (kj == qi)[None, None, :, :]

        y = F.scaled_dot_product_attention(
            q, k_attn, v_attn, attn_mask=attn_mask, dropout_p=drop, is_causal=is_causal
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y), new_kv


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.up_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.dropout(self.down_proj(x))


class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps):
        super().__init__()
        self.norm1 = RMSNorm(d_model, rms_eps)
        self.attn = GQACausalSelfAttention(d_model, n_heads, n_kv_heads, dropout)
        self.norm2 = RMSNorm(d_model, rms_eps)
        self.ff = SwiGLU(d_model, ffn_dim, dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        a, new_kv = self.attn(
            self.norm1(x), cos, sin, past_kv=past_kv, use_cache=use_cache, key_valid=key_valid
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.norm2(x))
        return x, new_kv


class EfficientDecoderLM(nn.Module):
    def __init__(
        self, vocab_size, d_model, n_layers, n_heads, n_kv_heads,
        ffn_dim, dropout, max_positions, rope_base, rms_eps
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryTable(d_model // n_heads, max_positions, rope_base)

        self.blocks = nn.ModuleList([
            EfficientDecoderBlock(d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps)
            for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model, rms_eps)

        self.apply(self._init_weights)
        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.o_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff.down_proj.weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward_hidden(self, input_ids, past_key_values=None, use_cache=False,
                       position_ids=None, key_valid=None):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        if past_len + T > self.max_positions:
            raise ValueError(f"context {past_len + T} exceeds max_positions={self.max_positions}")

        if position_ids is None:
            position_ids = torch.arange(past_len, past_len + T, device=input_ids.device)[None, :].expand(B, T)
        cos, sin = self.rope(position_ids)               # once for all layers (callers keep ids < max_positions)

        x = self.drop(self.token_emb(input_ids))
        new_cache = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(x, cos, sin, past_kv=past, use_cache=use_cache, key_valid=key_valid)
            if use_cache:
                new_cache.append(kv)
        return self.final_norm(x), new_cache

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)   # tied token/output embedding



def build_backbone():
    return EfficientDecoderLM(
        vocab_size=len(tokenizer), d_model=D_MODEL, n_layers=N_LAYERS,
        n_heads=N_HEADS, n_kv_heads=N_KV_HEADS, ffn_dim=FFN_DIM, dropout=DROPOUT,
        max_positions=MAX_POSITIONS, rope_base=ROPE_BASE, rms_eps=RMS_EPS,
    )

backbone=build_backbone()
obj=torch.load(MODEL4_CKPT,map_location="cpu",weights_only=False)
assert isinstance(obj,dict) and "model" in obj
missing,unexpected=backbone.load_state_dict(obj["model"],strict=False)
assert not missing and not unexpected,(missing,unexpected)
print("Strict Model4 load PASS | params=%.3fM"%(sum(p.numel() for p in backbone.parameters())/1e6))
print("checkpoint global_step:",obj.get("global_step"),"val_loss:",obj.get("val_loss"))
print("checkpoint config:",obj.get("config"))

# Freeze all, then unfreeze top five blocks + final norm.
for p in backbone.parameters(): p.requires_grad=False
for block in backbone.blocks[FROZEN_BLOCKS:]:
    for p in block.parameters(): p.requires_grad=True
for p in backbone.final_norm.parameters(): p.requires_grad=True

print("backbone trainable params:",sum(p.numel() for p in backbone.parameters() if p.requires_grad)/1e6,"M")


Strict Model4 load PASS | params=99.309M
checkpoint global_step: 16266 val_loss: 1.3664915735571412
checkpoint config: {'architecture': 'efficient_gqa_rope_swiglu_decoder', 'vocab_size': 24576, 'd_model': 640, 'n_layers': 17, 'n_heads': 10, 'n_kv_heads': 2, 'rope_base': 10000.0, 'ffn_dim': 2048, 'dropout': 0.0, 'max_positions': 2560, 'seq_len': 1536, 'prompt_p10_tokens': 1050, 'prompt_p95_tokens': 1139, 'expected_cont_tokens': 384, 'prompt_split_min': 1050, 'prompt_split_max': 1139}
backbone trainable params: 24.58304 M


In [8]:
# CELL 8 — Phase-5 hierarchical multilabel classifier
class StatuteHierarchicalClassifier(nn.Module):
    def __init__(self, backbone, d_model=640, proj_dim=384, n_labels=100, dropout=0.10):
        super().__init__()
        self.backbone=backbone
        self.proj=nn.Sequential(
            nn.Linear(2*d_model,proj_dim),
            nn.GELU(),
            nn.LayerNorm(proj_dim),
            nn.Dropout(dropout),
        )
        self.chunk_pos=nn.Parameter(torch.zeros(N_CHUNKS,proj_dim))
        nn.init.normal_(self.chunk_pos,std=0.02)
        self.attn=nn.Sequential(
            nn.Linear(proj_dim,128),nn.Tanh(),nn.Linear(128,1)
        )
        self.head=nn.Sequential(
            nn.Linear(2*proj_dim,proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim,n_labels),
        )

    def encode_flat_chunks(self, ids, valid):
        # ids/valid: (B*C,T). Lower 12 frozen blocks run without autograd.
        B,T=ids.shape
        position_ids=torch.arange(T,device=ids.device)[None,:].expand(B,T)
        cos,sin=self.backbone.rope(position_ids)

        # v2: inputs are RIGHT-padded and the backbone is causal, so every valid token only ever attends to
        # earlier valid tokens. The padding mask is therefore redundant for valid positions (pooling below
        # ignores padded positions anyway) and dropping it lets SDPA use its fast is_causal kernel.
        kv=None if FAST_CAUSAL_NO_MASK else valid
        with torch.no_grad():
            x=self.backbone.drop(self.backbone.token_emb(ids))
            for block in self.backbone.blocks[:FROZEN_BLOCKS]:
                x,_=block(x,cos,sin,past_kv=None,use_cache=False,key_valid=kv)
        x=x.detach()

        for block in self.backbone.blocks[FROZEN_BLOCKS:]:
            x,_=block(x,cos,sin,past_kv=None,use_cache=False,key_valid=kv)
        x=self.backbone.final_norm(x)

        mask=valid.unsqueeze(-1).to(x.dtype)
        mean=(x*mask).sum(1)/mask.sum(1).clamp_min(1.0)
        last_idx=valid.long().sum(1).clamp_min(1)-1
        last=x[torch.arange(B,device=x.device),last_idx]
        return self.proj(torch.cat([last,mean],-1))

    def forward(self,input_ids,valid):
        # (B,C,T)
        B,C,T=input_ids.shape
        z=self.encode_flat_chunks(input_ids.reshape(B*C,T),valid.reshape(B*C,T)).reshape(B,C,-1)
        z=z+self.chunk_pos[None,:,:]
        scores=self.attn(z).squeeze(-1)
        weights=torch.softmax(scores,-1)
        attn_pool=(z*weights.unsqueeze(-1)).sum(1)
        max_pool=z.max(1).values
        logits=self.head(torch.cat([attn_pool,max_pool],-1))
        return logits,weights

model=StatuteHierarchicalClassifier(backbone).to(DEVICE)

total=sum(p.numel() for p in model.parameters())
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total={total/1e6:.3f}M trainable={trainable/1e6:.3f}M ({100*trainable/total:.1f}%)")
assert 0<trainable<total


total=100.187M trainable=25.460M (25.4%)


In [9]:
# CELL 9 — datasets/loaders
class LSIDataset(Dataset):
    def __init__(self,records,tok,lens,labeled=True):
        self.records=records; self.tok=tok; self.lens=lens; self.labeled=labeled
    def __len__(self): return len(self.records)
    def __getitem__(self,i):
        ids=torch.from_numpy(np.array(self.tok[i],dtype=np.int64,copy=True))
        lens=torch.from_numpy(np.array(self.lens[i],dtype=np.int64,copy=True))
        valid=torch.arange(CHUNK_LEN)[None,:] < lens[:,None]
        out={"input_ids":ids,"valid":valid,"id":str(self.records[i]["id"])}
        if self.labeled:
            y=torch.zeros(N_LABELS,dtype=torch.float32)
            y[self.records[i]["labels"]]=1.0
            out["labels"]=y
        return out

def make_loader(records,tok,lens,batch,shuffle,labeled=True,seed=SEED):
    g=torch.Generator(); g.manual_seed(seed)
    return DataLoader(
        LSIDataset(records,tok,lens,labeled),
        batch_size=batch,shuffle=shuffle,num_workers=NUM_WORKERS,
        pin_memory=True,persistent_workers=(NUM_WORKERS>0),
        generator=g,
    )

train_loader=make_loader(train_records,train_tok,train_lens,MICRO_BATCH_SIZE,True,True)
official_loader=make_loader(dev_raw,official_tok,official_lens,DEV_BATCH_SIZE,False,True)
test_loader=make_loader(test_records,test_tok,test_lens,DEV_BATCH_SIZE,False,False)

print("loader batches:",len(train_loader),len(official_loader),len(test_loader))


loader batches: 21375 1273 1628


In [10]:
# CELL 10 — exact evaluator logic + threshold tuning
def multilabel_metrics(y_true,y_pred):
    return {
        "macro_f1":float(f1_score(y_true,y_pred,average="macro",zero_division=0)),
        "micro_f1":float(f1_score(y_true,y_pred,average="micro",zero_division=0)),
        "subset_accuracy":float(accuracy_score(y_true,y_pred)),
    }

def apply_thresholds(prob, thresholds, ensure_nonempty=True):
    """Apply per-label thresholds and optionally guarantee >=1 prediction/document."""
    thr=np.asarray(thresholds,dtype=np.float32)
    pred=prob>=thr[None,:]
    if ensure_nonempty:
        empty=np.flatnonzero(~pred.any(1))
        if len(empty):
            # Pick the label closest to clearing its own threshold.
            best=np.argmax(prob[empty]/np.maximum(thr[None,:],1e-6),axis=1)
            pred[empty,best]=True
    return pred

# ---------- v2: vectorised threshold utilities ----------
def macro_f1_fast(y,pred):
    """Identical to sklearn macro-F1 with zero_division=0 (verified in testing), but ~100x faster."""
    yb=y.astype(bool); pb=pred.astype(bool)
    tp=(yb&pb).sum(0).astype(np.float64)
    den=yb.sum(0)+pb.sum(0)
    return float(np.where(den>0,2*tp/np.maximum(den,1),0.0).mean())

def f1_curves(y,prob,grid=THR_GRID):
    """(G,L) matrix: per-label F1 at each grid threshold."""
    yb=y.astype(bool); P=yb.sum(0).astype(np.float64)
    out=np.zeros((len(grid),y.shape[1]),dtype=np.float64)
    for gi,t in enumerate(grid):
        pr=prob>=t
        tp=(pr&yb).sum(0); den=pr.sum(0)+P
        out[gi]=np.where(den>0,2*tp/np.maximum(den,1),0.0)
    return out

def smooth_curves(c):
    """3-point moving average along the threshold axis -> picks the centre of a plateau, not a lucky spike."""
    p=np.pad(c,((1,1),(0,0)),mode="edge")
    return (p[:-2]+p[1:-1]+p[2:])/3.0

def fit_threshold_strategies(y,prob):
    """Return {strategy_name: thresholds(L,)} fitted on (y,prob)."""
    c=f1_curves(y,prob)
    g=float(THR_GRID[int(np.argmax(c.mean(1)))])
    per=THR_GRID[np.argmax(smooth_curves(c),axis=0)].astype(np.float32)
    P=y.sum(0).astype(np.float64)
    out={"global":np.full(y.shape[1],g,dtype=np.float32),"per_label":per}
    for tau in SHRINK_TAUS:
        w=P/(P+tau)
        out[f"shrunk_tau_{int(tau)}"]=(w*per+(1-w)*g).astype(np.float32)
    return out,g,per

def optimize_thresholds(y,prob):
    """Choose the strategy by cross-fitted macro-F1 (fit on half A, score on half B, and vice-versa),
    then refit the winner on ALL given data. Returns the in-sample score too, but selection uses CV."""
    full,g,per=fit_threshold_strategies(y,prob)
    n=len(y); cv={k:[] for k in full}
    for rep_i in range(CV_REPEATS):
        perm=np.random.RandomState(100+rep_i).permutation(n)
        a,b=perm[:n//2],perm[n//2:]
        for tr,te in ((a,b),(b,a)):
            strat,_,_=fit_threshold_strategies(y[tr],prob[tr])
            for k,thr in strat.items():
                cv[k].append(macro_f1_fast(y[te],apply_thresholds(prob[te],thr,ensure_nonempty=True)))
    cv={k:float(np.mean(v)) for k,v in cv.items()}
    name=max(cv,key=cv.get)
    thr=full[name]
    return {"name":name,"thresholds":thr,"cv_macro_f1":cv[name],
            "insample_macro_f1":macro_f1_fast(y,apply_thresholds(prob,thr,ensure_nonempty=True)),
            "cv_all":cv},g,per

@torch.inference_mode()
def predict_loader(model,loader):
    model.eval()
    ids=[]; probs=[]; ys=[]; atts=[]
    for b in loader:
        x=b["input_ids"].to(DEVICE,non_blocking=True)
        v=b["valid"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda" and USE_AMP)):
            lg,a=model(x,v)
        assert torch.isfinite(lg).all()
        ids.extend(list(b["id"]))
        probs.append(torch.sigmoid(lg.float()).cpu().numpy())
        atts.append(a.float().cpu().numpy())
        if "labels" in b: ys.append(b["labels"].numpy().astype(np.uint8))
    return ids,np.concatenate(probs), (np.concatenate(ys) if ys else None), np.concatenate(atts)

def make_pred_records(ids,prob,thresholds,ensure_nonempty=True):
    binary=apply_thresholds(prob,thresholds,ensure_nonempty=ensure_nonempty)
    return [{"id":str(i),"pred":[int(x) for x in np.flatnonzero(binary[k])]} for k,i in enumerate(ids)]

# Exact supplied evaluator behavior, reproduced in-process.
def exact_phase5_score(truth_records,pred_records):
    t={}; max_label=-1
    for n,r in enumerate(truth_records):
        assert isinstance(r,dict) and "id" in r and "gold" in r
        assert isinstance(r["id"],str) and r["id"].strip() and isinstance(r["gold"],list)
        assert r["id"] not in t
        seen=set()
        for label in r["gold"]:
            assert not isinstance(label,bool) and isinstance(label,int) and label>=0 and label not in seen
            seen.add(label); max_label=max(max_label,label)
        t[r["id"]]=r["gold"]
    nlab=max_label+1
    p={}
    for n,r in enumerate(pred_records):
        assert isinstance(r,dict) and "id" in r and "pred" in r
        assert isinstance(r["id"],str) and isinstance(r["pred"],list) and r["id"] not in p
        seen=set()
        for x in r["pred"]:
            assert not isinstance(x,bool) and isinstance(x,int) and 0<=x<nlab and x not in seen
            seen.add(x)
        p[r["id"]]=r["pred"]
    assert set(t)==set(p)
    yt=np.zeros((len(t),nlab),dtype=np.uint8); yp=np.zeros((len(t),nlab),dtype=np.uint8)
    for i,k in enumerate(t):
        for x in t[k]: yt[i,x]=1
        for x in p[k]: yp[i,x]=1
    return multilabel_metrics(yt,yp)

print("threshold utilities ready")


threshold utilities ready


In [12]:
# CELL 11 — optimizer/scheduler and main partial-FT training
pos_weight_t=torch.tensor(POS_WEIGHT,dtype=torch.float32,device=DEVICE)

backbone_params=[]
head_params=[]
for n,p in model.named_parameters():
    if not p.requires_grad: continue
    if n.startswith("backbone."): backbone_params.append(p)
    else: head_params.append(p)

optimizer=torch.optim.AdamW([
    {"params":backbone_params,"lr":BACKBONE_LR,"weight_decay":WEIGHT_DECAY},
    {"params":head_params,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY},
],betas=(0.9,0.999),eps=1e-8)

updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
total_updates=updates_per_epoch*EPOCHS
warmup=max(1,int(WARMUP_RATIO*total_updates))

def lr_lambda(step):
    if step<warmup: return (step+1)/warmup
    q=(step-warmup)/max(1,total_updates-warmup)
    return max(0.05,0.5*(1+math.cos(math.pi*min(1.0,q))))

scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda)
scaler=torch.amp.GradScaler("cuda",enabled=(DEVICE.type=="cuda" and USE_AMP))

BEST=ROOT/"best_partialft_lsi_v3.pt"
HISTORY=ROOT/"history.json"
eval_points=sorted(set(
    min(len(train_loader),max(1,round(len(train_loader)*k/EVALS_PER_EPOCH)))
    for k in range(1,EVALS_PER_EPOCH+1)
))
print("eval batches per epoch:",eval_points)
print("total optimizer updates:",total_updates)

# ---------- v2: EMA of trainable weights ----------
class EMA:
    def __init__(self,model,decay):
        self.decay=decay
        self.params={n:p for n,p in model.named_parameters() if p.requires_grad}
        self.shadow={n:p.detach().clone() for n,p in self.params.items()}
        self.n=0
    @torch.no_grad()
    def update(self):
        self.n+=1
        d=min(self.decay,(1.0+self.n)/(10.0+self.n))     # warm-up so early EMA is not dominated by init
        for n,p in self.params.items():
            self.shadow[n].mul_(d).add_(p.detach(),alpha=1.0-d)
    @contextlib.contextmanager
    def applied(self):
        backup={n:p.detach().clone() for n,p in self.params.items()}
        with torch.no_grad():
            for n,p in self.params.items(): p.copy_(self.shadow[n])
        try: yield
        finally:
            with torch.no_grad():
                for n,p in self.params.items(): p.copy_(backup[n])

ema=EMA(model,EMA_DECAY)

best_score=-1.0     # v2: selection metric = cross-fitted clean-DEV Macro-F1 (honest), not in-sample
history=[]
global_update=0
optimizer.zero_grad(set_to_none=True)

def evaluate_checkpoint(epoch,batch_idx,tag):
    global best_score
    # v2: score official DEV ONCE; clean DEV is a subset of it.
    oids,oprob,oy,oatt=predict_loader(model,official_loader)
    cprob=oprob[CLEAN_IDX]; cy=oy[CLEAN_IDX]; cids=[oids[i] for i in CLEAN_IDX]; catt=oatt[CLEAN_IDX]
    tune,g,per=optimize_thresholds(cy,cprob)
    clean_pred=make_pred_records(cids,cprob,tune["thresholds"])
    exact_clean=exact_phase5_score(clean_truth,clean_pred)
    official_pred=make_pred_records(oids,oprob,tune["thresholds"])
    exact_official=exact_phase5_score(official_truth,official_pred)

    row={
        "epoch":epoch,"batch":batch_idx,"weights":tag,
        "epoch_frac":(epoch-1)+batch_idx/len(train_loader),
        "strategy":tune["name"],"global_threshold":g,
        "cv_macro_f1":tune["cv_macro_f1"],"insample_macro_f1":tune["insample_macro_f1"],
        "cv_all":tune["cv_all"],
        "clean":exact_clean,"official":exact_official,
        "mean_pred_labels_clean":float(np.mean([len(r["pred"]) for r in clean_pred])),
        "mean_gold_labels_clean":float(cy.sum(1).mean()),
        "attention_mean_clean":catt.mean(0).tolist(),
    }
    history.append(row)
    json.dump(history,open(HISTORY,"w"),indent=2)
    print(f"[{tag}] ep={row['epoch_frac']:.2f} strat={tune['name']} CV-macroF1={tune['cv_macro_f1']:.4f} "
          f"clean macro/micro/subset={exact_clean['macro_f1']:.4f}/{exact_clean['micro_f1']:.4f}/{exact_clean['subset_accuracy']:.4f} "
          f"official macro={exact_official['macro_f1']:.4f} pred/gold labels={row['mean_pred_labels_clean']:.2f}/{row['mean_gold_labels_clean']:.2f}",flush=True)

    if tune["cv_macro_f1"]>best_score:
        best_score=tune["cv_macro_f1"]
        atomic_torch_save({
            "model_state":{k:v.detach().cpu() for k,v in model.state_dict().items()},   # EMA weights when tag=="ema"
            "weights_tag":tag,
            "thresholds":np.asarray(tune["thresholds"],dtype=np.float32),
            "threshold_strategy":tune["name"],
            "global_threshold":g,
            "per_label_thresholds":np.asarray(per,dtype=np.float32),
            "cv_macro_f1":tune["cv_macro_f1"],
            "clean_metrics":exact_clean,
            "official_metrics":exact_official,
            "epoch_frac":row["epoch_frac"],
            "model4_sha256":sha256_file(MODEL4_CKPT),
            "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
            "dataset_sha256":DATA_SHA256,
            "config":{
                "n_labels":N_LABELS,"n_chunks":N_CHUNKS,"chunk_len":CHUNK_LEN,
                "frozen_blocks":FROZEN_BLOCKS,"backbone_lr":BACKBONE_LR,"head_lr":HEAD_LR,
                "epochs":EPOCHS,"ema_decay":EMA_DECAY,
                "pos_weight":"sqrt(raw) clipped [1,10]",
            },
        },BEST)
        print("✅ NEW BEST cross-fitted clean-DEV Macro-F1:",round(best_score,5),f"({tag})")

print("If CUDA OOM occurs before the first batch completes: set MICRO_BATCH_SIZE=1, GRAD_ACCUM_STEPS=16, then restart from Cell 9.")

for epoch in range(1,EPOCHS+1):
    model.train()
    t0=time.time(); running=0.0; good=0
    for bi,b in enumerate(train_loader,1):
        x=b["input_ids"].to(DEVICE,non_blocking=True)
        v=b["valid"].to(DEVICE,non_blocking=True)
        y=b["labels"].to(DEVICE,non_blocking=True)

        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda" and USE_AMP)):
            logits,_=model(x,v)
            loss=F.binary_cross_entropy_with_logits(logits,y,pos_weight=pos_weight_t)
            scaled_loss=loss/GRAD_ACCUM_STEPS

        if torch.isfinite(loss):
            scaler.scale(scaled_loss).backward(); good+=1
            running+=float(loss.detach())
        else:
            print("⚠️ non-finite loss at batch",bi)

        if bi%GRAD_ACCUM_STEPS==0 or bi==len(train_loader):
            if good:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],GRAD_CLIP)
                old_scale=float(scaler.get_scale())
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if float(scaler.get_scale())>=old_scale:
                    scheduler.step(); global_update+=1
                    ema.update()                      # v2: EMA only advances on real optimizer steps
            else:
                optimizer.zero_grad(set_to_none=True)
            good=0

        if bi%250==0:
            elapsed=(time.time()-t0)/60
            eta=elapsed*(len(train_loader)-bi)/bi
            print(f"epoch={epoch} batch={bi}/{len(train_loader)} loss={running/max(1,bi):.4f} elapsed={elapsed:.1f}m ETA={eta:.1f}m",flush=True)

        if bi in eval_points:
            with ema.applied():                       # v2: evaluate / select on EMA weights
                evaluate_checkpoint(epoch,bi,"ema")
            if bi==len(train_loader):                 # also score the raw weights once per epoch
                evaluate_checkpoint(epoch,bi,"raw")
            model.train()

assert BEST.exists()
print("✅ training complete | best cross-fitted clean DEV Macro-F1:",best_score)


eval batches per epoch: [7125, 14250, 21375]
total optimizer updates: 8016
If CUDA OOM occurs before the first batch completes: set MICRO_BATCH_SIZE=1, GRAD_ACCUM_STEPS=16, then restart from Cell 9.


RuntimeError: DataLoader worker (pid(s) 166, 167) exited unexpectedly

In [13]:
BEST = Path("/kaggle/input/datasets/aaryant23/phase2-model4/best_partialft_lsi_v3.pt")
print(BEST, BEST.exists())

/kaggle/input/datasets/aaryant23/phase2-model4/best_partialft_lsi_v3.pt True


In [14]:
import gc
try: del train_loader
except NameError: pass
gc.collect()

official_loader=make_loader(dev_raw,official_tok,official_lens,DEV_BATCH_SIZE,False,True)
test_loader=make_loader(test_records,test_tok,test_lens,DEV_BATCH_SIZE,False,False)
print("loaders rebuilt:",len(official_loader),len(test_loader))

loaders rebuilt: 1273 1628


In [15]:
# CELL 12 — reload BEST and produce final held-out diagnostics
best=torch.load(BEST,map_location="cpu",weights_only=False)
model.load_state_dict(best["model_state"],strict=True)
model.to(DEVICE); model.eval()
thresholds=np.asarray(best["thresholds"],dtype=np.float32)

print("BEST epoch_frac:",best["epoch_frac"],"| weights:",best["weights_tag"])
print("threshold strategy:",best["threshold_strategy"],"| CV macro-F1:",best["cv_macro_f1"])
print("clean metrics:",best["clean_metrics"])
print("official metrics:",best["official_metrics"])
print("threshold min/median/max:",thresholds.min(),np.median(thresholds),thresholds.max())

# Re-score after reload (must reproduce the saved numbers) and save reusable DEV probabilities for ensembling.
dids,dprob,dy,datt=predict_loader(model,official_loader)
re_pred=make_pred_records([dids[i] for i in CLEAN_IDX],dprob[CLEAN_IDX],thresholds)
re_score=exact_phase5_score(clean_truth,re_pred)
print("re-scored clean DEV after reload:",re_score)
assert abs(re_score["macro_f1"]-best["clean_metrics"]["macro_f1"])<5e-3,"reload mismatch!"
np.savez_compressed(ROOT/"dev_official_probs.npz",id=np.asarray(dids),prob=dprob,y=dy,
                    clean_idx=CLEAN_IDX,thresholds=thresholds)
print("mean attention [HEAD,MIDDLE,EVIDENCE,TAIL]:",datt[CLEAN_IDX].mean(0))
print("mean predicted labels:",apply_thresholds(dprob[CLEAN_IDX],thresholds,True).sum(1).mean(),"gold:",dy[CLEAN_IDX].sum(1).mean())

# Per-label diagnostic for the report: worst labels on clean DEV.
from sklearn.metrics import f1_score as _f1
cy_=dy[CLEAN_IDX]; cp_=apply_thresholds(dprob[CLEAN_IDX],thresholds,True).astype(np.uint8)
per_f1=np.array([_f1(cy_[:,j],cp_[:,j],zero_division=0) for j in range(N_LABELS)])
worst=np.argsort(per_f1)[:10]
print("10 weakest labels (label, F1, train_count):",[(int(j),round(float(per_f1[j]),3),int(train_counts[j])) for j in worst])
json.dump({"per_label_f1":per_f1.tolist()},open(ROOT/"per_label_f1.json","w"))


BEST epoch_frac: 2.0 | weights: ema
threshold strategy: shrunk_tau_25 | CV macro-F1: 0.3521102049841443
clean metrics: {'macro_f1': 0.3611655181807631, 'micro_f1': 0.46368238565240993, 'subset_accuracy': 0.03408526284408228}
official metrics: {'macro_f1': 0.3606767810264691, 'micro_f1': 0.46320573326341546, 'subset_accuracy': 0.03408309596306846}
threshold min/median/max: 0.30491105 0.5482936 0.76551723
re-scored clean DEV after reload: {'macro_f1': 0.3611655181807631, 'micro_f1': 0.46368238565240993, 'subset_accuracy': 0.03408526284408228}
mean attention [HEAD,MIDDLE,EVIDENCE,TAIL]: [0.31612146 0.15726568 0.18127699 0.34533507]
mean predicted labels: 5.0592268707144985 gold: 3.9115571897048595
10 weakest labels (label, F1, train_count): [(2, 0.05, 367), (8, 0.058, 349), (0, 0.063, 343), (65, 0.084, 351), (84, 0.12, 272), (41, 0.126, 821), (99, 0.126, 674), (52, 0.133, 1084), (76, 0.163, 530), (15, 0.163, 336)]


In [16]:
# CELL — write the exact official Phase-5 evaluator supplied by the course
EVAL_SCRIPT = ROOT / "evaluate_phase5.py"
EVAL_SCRIPT.write_text('#!/usr/bin/env python3\n"""Local evaluator for Phase 5.\n\nTruth JSON: a list of objects, [{"id":"...", "gold":[0,2,5,...]}, ...]\nPrediction JSON: [{"id":"...", "pred":[0,2,5,...]}, ...]\n"""\nimport argparse,json,sys\nimport numpy as np\nfrom sklearn.metrics import accuracy_score, f1_score\n\ndef read_truth(path):\n    try:\n        x=json.load(open(path,encoding="utf-8"))\n    except json.JSONDecodeError as e:\n        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")\n    if not isinstance(x,list) or not x: raise ValueError(\'Truth must be a non-empty JSON list.\')\n    out={}; max_label=-1\n    for n,r in enumerate(x):\n        if not isinstance(r,dict) or \'id\' not in r or \'gold\' not in r: raise ValueError(f\'Truth item {n} must contain id and gold.\')\n        if not isinstance(r[\'id\'],str) or not r[\'id\'].strip() or not isinstance(r[\'gold\'],list): raise ValueError(f\'Invalid truth item {n}.\')\n        if r[\'id\'] in out: raise ValueError(f\'Duplicate truth ID: {r["id"]}\')\n        seen=set()\n        for label in r[\'gold\']:\n            if isinstance(label,bool) or not isinstance(label,int) or label<0 or label in seen: raise ValueError(f\'Invalid/duplicate label in truth item {n}.\')\n            seen.add(label); max_label=max(max_label,label)\n        out[r[\'id\']]=r[\'gold\']\n    return out,max_label+1\n\ndef read_pred(path,nlab):\n    x=json.load(open(path,encoding=\'utf-8\'))\n    if not isinstance(x,list) or not x: raise ValueError(\'Prediction must be a non-empty JSON list.\')\n    out={}\n    for n,r in enumerate(x):\n        if not isinstance(r,dict) or \'id\' not in r or \'pred\' not in r: raise ValueError(f\'Prediction item {n} must contain id and pred.\')\n        if not isinstance(r[\'id\'],str) or not isinstance(r[\'pred\'],list) or r[\'id\'] in out: raise ValueError(f\'Invalid or duplicate prediction item {n}.\')\n        seen=set()\n        for x in r[\'pred\']:\n            if isinstance(x,bool) or not isinstance(x,int) or x<0 or x>=nlab or x in seen: raise ValueError(f\'Invalid/duplicate label in prediction item {n}.\')\n            seen.add(x)\n        out[r[\'id\']]=r[\'pred\']\n    return out\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 5 multilabel predictions on your own held-out split.\')\n    ap.add_argument(\'--truth\',required=True); ap.add_argument(\'--prediction\',required=True); a=ap.parse_args()\n    t,n=read_truth(a.truth); p=read_pred(a.prediction,n)\n    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")\n    yt=np.zeros((len(t),n),dtype=np.uint8); yp=np.zeros((len(t),n),dtype=np.uint8)\n    for i,k in enumerate(t):\n        for x in t[k]: yt[i,x]=1\n        for x in p[k]: yp[i,x]=1\n    print(f\'Examples: {len(t)}\'); print(f\'Labels: {n}\')\n    print(f\'Macro-F1: {f1_score(yt,yp,average="macro",zero_division=0):.6f}\')\n    micro=f1_score(yt,yp,average=\'micro\',zero_division=0)\n    subset=accuracy_score(yt,yp)\n    print(f\'Micro-F1: {micro:.6f}\')\n    print(f\'Subset Accuracy: {subset:.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\',file=sys.stderr); sys.exit(1)\n', encoding="utf-8")
print("✅ wrote exact evaluator:", EVAL_SCRIPT)


✅ wrote exact evaluator: /kaggle/working/phase5_exp1_model4_partialft_v5/evaluate_phase5.py


In [17]:
# CELL — literal evaluator-script test on BEST official DEV predictions
# Writes real JSON files and invokes evaluate_phase5.py as a subprocess.

dev_pred_records = make_pred_records(dids, dprob, thresholds, ensure_nonempty=True)

DEV_PRED_JSON = ROOT / "dev_prediction_best.json"
DEV_TRUTH_JSON = ROOT / "dev_truth_official.json"

json.dump(dev_pred_records, open(DEV_PRED_JSON, "w", encoding="utf-8"), ensure_ascii=False)
assert DEV_TRUTH_JSON.exists(), DEV_TRUTH_JSON

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--truth", str(DEV_TRUTH_JSON),
    "--prediction", str(DEV_PRED_JSON),
]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)

print("\n--- evaluate_phase5.py STDOUT ---")
print(proc.stdout)
if proc.stderr.strip():
    print("--- evaluate_phase5.py STDERR ---")
    print(proc.stderr)

assert proc.returncode == 0, "Official evaluator script FAILED on DEV prediction JSON."

m = re.search(r"Macro-F1:\s*([0-9.]+)", proc.stdout)
assert m, "Could not parse Macro-F1 from evaluator output."
script_macro = float(m.group(1))
internal_macro = float(best["official_metrics"]["macro_f1"])

print("Evaluator Macro-F1 :", script_macro)
print("Internal Macro-F1  :", internal_macro)
assert abs(script_macro - internal_macro) <= 5e-6, (
    f"Evaluator/internal mismatch: {script_macro} vs {internal_macro}"
)
print("✅ exact evaluator script agrees with internal scoring.")


Running: /usr/bin/python3 /kaggle/working/phase5_exp1_model4_partialft_v5/evaluate_phase5.py --truth /kaggle/working/phase5_exp1_model4_partialft_v5/dev_truth_official.json --prediction /kaggle/working/phase5_exp1_model4_partialft_v5/dev_prediction_best.json

--- evaluate_phase5.py STDOUT ---
Examples: 10181
Labels: 100
Macro-F1: 0.360677
Micro-F1: 0.463206
Subset Accuracy: 0.034083

Evaluator Macro-F1 : 0.360677
Internal Macro-F1  : 0.3606767810264691
✅ exact evaluator script agrees with internal scoring.


In [18]:
# CELL — TEST inference + Codabench submission ZIP + ZIP validation
tids,tprob,_,tatt=predict_loader(model,test_loader)
pred_records=make_pred_records(tids,tprob,thresholds,ensure_nonempty=True)

assert len(pred_records)==len(test_records)
assert len({r["id"] for r in pred_records})==len(test_records)
assert {r["id"] for r in pred_records}=={str(r["id"]) for r in test_records}
for r in pred_records:
    assert len(r["pred"])==len(set(r["pred"]))
    assert all(isinstance(x,int) and not isinstance(x,bool) and 0<=x<N_LABELS for x in r["pred"])

PRED=ROOT/"prediction.json"
json.dump(pred_records,open(PRED,"w",encoding="utf-8"),ensure_ascii=False)

ZIP=ROOT/"submission_exp1_HELDOUT.zip"
with zipfile.ZipFile(ZIP,"w",compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED,arcname="prediction.json")

print("TEST mean predicted labels/document:",np.mean([len(r["pred"]) for r in pred_records]))
print("TEST empty predictions:",sum(len(r["pred"])==0 for r in pred_records))
print("TEST attention mean [HEAD,MIDDLE,EVIDENCE,TAIL]:",tatt.mean(0))
print("✅ submission:",ZIP)

np.savez_compressed(
    ROOT/"heldout_test_probs.npz",
    id=np.asarray(tids), prob=tprob, thresholds=thresholds
)
print("✅ saved heldout TEST probabilities for ensembling")

# Validate actual prediction.json inside the ZIP using evaluator read_pred().
ZIP_CHECK_DIR=ROOT/"zip_check"
if ZIP_CHECK_DIR.exists():
    shutil.rmtree(ZIP_CHECK_DIR)
ZIP_CHECK_DIR.mkdir(parents=True,exist_ok=True)

with zipfile.ZipFile(ZIP,"r") as z:
    names=z.namelist()
    print("ZIP contents:",names)
    assert names==["prediction.json"], f"Unexpected ZIP contents: {names}"
    z.extract("prediction.json",ZIP_CHECK_DIR)

ZIP_PRED=ZIP_CHECK_DIR/"prediction.json"
assert ZIP_PRED.exists()

import importlib.util
spec=importlib.util.spec_from_file_location("evaluate_phase5",EVAL_SCRIPT)
evalmod=importlib.util.module_from_spec(spec)
spec.loader.exec_module(evalmod)

zip_pred_dict=evalmod.read_pred(str(ZIP_PRED),N_LABELS)

expected_test_ids={str(r["id"]) for r in test_records}
assert set(zip_pred_dict)==expected_test_ids, (
    f"ZIP prediction ID mismatch: missing={len(expected_test_ids-set(zip_pred_dict))}, "
    f"extra={len(set(zip_pred_dict)-expected_test_ids)}"
)

print("✅ prediction.json inside ZIP passes exact evaluator read_pred().")
print("✅ ZIP contains every TEST id exactly once.")
print("✅ label type/range/duplicate-label checks all pass.")


TEST mean predicted labels/document: 4.986942161456333
TEST empty predictions: 0
TEST attention mean [HEAD,MIDDLE,EVIDENCE,TAIL]: [0.31485462 0.16007774 0.17938766 0.34567952]
✅ submission: /kaggle/working/phase5_exp1_model4_partialft_v5/submission_exp1_HELDOUT.zip
✅ saved heldout TEST probabilities for ensembling
ZIP contents: ['prediction.json']
✅ prediction.json inside ZIP passes exact evaluator read_pred().
✅ ZIP contains every TEST id exactly once.
✅ label type/range/duplicate-label checks all pass.


In [20]:
class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.params = {
            n: p
            for n, p in model.named_parameters()
            if p.requires_grad
        }
        self.shadow = {
            n: p.detach().clone()
            for n, p in self.params.items()
        }
        self.n = 0

    @torch.no_grad()
    def update(self):
        self.n += 1
        d = min(
            self.decay,
            (1.0 + self.n) / (10.0 + self.n)
        )

        for n, p in self.params.items():
            self.shadow[n].mul_(d).add_(
                p.detach(),
                alpha=1.0 - d
            )

    @contextlib.contextmanager
    def applied(self):
        backup = {
            n: p.detach().clone()
            for n, p in self.params.items()
        }

        with torch.no_grad():
            for n, p in self.params.items():
                p.copy_(self.shadow[n])

        try:
            yield
        finally:
            with torch.no_grad():
                for n, p in self.params.items():
                    p.copy_(backup[n])

print("EMA class loaded")

EMA class loaded


In [21]:
# CELL 14 — proper REFIT on TRAIN + official DEV, starting fresh from Phase-1 Model4
# The selected heldout model remains untouched. Refit is a separate model/submission.

if not RUN_REFIT:
    print("RUN_REFIT=False — heldout submission is ready; refit/ensemble are skipped.")
    REFIT_READY=False
else:
    print("="*90)
    print("REFIT: fresh Phase-1 Model4 -> TRAIN + DEV")
    print("="*90)

    # ------------------------------------------------------------------
    # 1) Rebuild a fresh task model from the same Phase-1 checkpoint.
    # ------------------------------------------------------------------
    random.seed(REFIT_SEED)
    np.random.seed(REFIT_SEED)
    torch.manual_seed(REFIT_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(REFIT_SEED)

    refit_backbone=build_backbone()
    refit_obj=torch.load(MODEL4_CKPT,map_location="cpu",weights_only=False)
    miss,unexp=refit_backbone.load_state_dict(refit_obj["model"],strict=False)
    assert not miss and not unexp,(miss,unexp)

    for p in refit_backbone.parameters():
        p.requires_grad=False
    for block in refit_backbone.blocks[FROZEN_BLOCKS:]:
        for p in block.parameters():
            p.requires_grad=True
    for p in refit_backbone.final_norm.parameters():
        p.requires_grad=True

    refit_model=StatuteHierarchicalClassifier(refit_backbone).to(DEVICE)

    # ------------------------------------------------------------------
    # 2) TRAIN + official DEV dataset. Use existing token caches; no retokenization.
    # ------------------------------------------------------------------
    refit_ds=ConcatDataset([
        LSIDataset(train_records,train_tok,train_lens,labeled=True),
        LSIDataset(dev_raw,official_tok,official_lens,labeled=True),
    ])

    refit_gen=torch.Generator()
    refit_gen.manual_seed(REFIT_SEED)
    refit_loader=DataLoader(
        refit_ds,batch_size=MICRO_BATCH_SIZE,shuffle=True,
        num_workers=NUM_WORKERS,pin_memory=True,
        persistent_workers=(NUM_WORKERS>0),generator=refit_gen,
    )

    # Recompute class weights on TRAIN+DEV.
    refit_counts=np.zeros(N_LABELS,dtype=np.int64)
    for r in train_records:
        refit_counts[r["labels"]]+=1
    for r in dev_raw:
        refit_counts[r["labels"]]+=1
    refit_N=len(train_records)+len(dev_raw)
    refit_raw_pw=(refit_N-refit_counts)/refit_counts
    REFIT_POS_WEIGHT=np.clip(np.sqrt(refit_raw_pw),1.0,10.0).astype(np.float32)
    refit_pos_weight_t=torch.tensor(REFIT_POS_WEIGHT,dtype=torch.float32,device=DEVICE)

    # Match the selected heldout training duration in data passes.
    # Example: best_epoch_frac=1.67 -> refit sees ~1.67 passes through TRAIN+DEV.
    selected_passes=max(0.20,float(best["epoch_frac"]))
    refit_batches_target=max(1,int(round(selected_passes*len(refit_loader))))
    refit_updates_target=max(1,math.ceil(refit_batches_target/GRAD_ACCUM_STEPS))
    refit_warmup=max(1,int(WARMUP_RATIO*refit_updates_target))

    print("TRAIN+DEV examples:",len(refit_ds))
    print("selected heldout passes:",selected_passes)
    print("refit target batches:",refit_batches_target)
    print("refit target optimizer updates:",refit_updates_target)

    refit_backbone_params=[]
    refit_head_params=[]
    for n,p in refit_model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("backbone."):
            refit_backbone_params.append(p)
        else:
            refit_head_params.append(p)

    refit_optimizer=torch.optim.AdamW([
        {"params":refit_backbone_params,"lr":BACKBONE_LR,"weight_decay":WEIGHT_DECAY},
        {"params":refit_head_params,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY},
    ],betas=(0.9,0.999),eps=1e-8)

    def refit_lr_lambda(step):
        if step<refit_warmup:
            return (step+1)/refit_warmup
        q=(step-refit_warmup)/max(1,refit_updates_target-refit_warmup)
        return max(0.05,0.5*(1+math.cos(math.pi*min(1.0,q))))

    refit_scheduler=torch.optim.lr_scheduler.LambdaLR(refit_optimizer,refit_lr_lambda)
    refit_scaler=torch.amp.GradScaler("cuda",enabled=(DEVICE.type=="cuda" and USE_AMP))
    refit_ema=EMA(refit_model,EMA_DECAY)

    refit_optimizer.zero_grad(set_to_none=True)
    refit_model.train()
    seen_batches=0
    refit_updates=0
    running=0.0
    t0=time.time()

    while seen_batches < refit_batches_target:
        for b in refit_loader:
            if seen_batches>=refit_batches_target:
                break
            seen_batches+=1

            x=b["input_ids"].to(DEVICE,non_blocking=True)
            v=b["valid"].to(DEVICE,non_blocking=True)
            y=b["labels"].to(DEVICE,non_blocking=True)

            with torch.amp.autocast(
                "cuda",dtype=torch.float16,
                enabled=(DEVICE.type=="cuda" and USE_AMP)
            ):
                logits,_=refit_model(x,v)
                loss=F.binary_cross_entropy_with_logits(
                    logits,y,pos_weight=refit_pos_weight_t
                )
                scaled_loss=loss/GRAD_ACCUM_STEPS

            assert torch.isfinite(loss),f"non-finite refit loss at batch {seen_batches}"
            refit_scaler.scale(scaled_loss).backward()
            running+=float(loss.detach())

            do_step=(seen_batches%GRAD_ACCUM_STEPS==0) or (seen_batches==refit_batches_target)
            if do_step:
                refit_scaler.unscale_(refit_optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in refit_model.parameters() if p.requires_grad],GRAD_CLIP
                )
                old_scale=float(refit_scaler.get_scale())
                refit_scaler.step(refit_optimizer)
                refit_scaler.update()
                refit_optimizer.zero_grad(set_to_none=True)

                if float(refit_scaler.get_scale())>=old_scale:
                    refit_scheduler.step()
                    refit_ema.update()
                    refit_updates+=1

            if seen_batches%250==0 or seen_batches==refit_batches_target:
                elapsed=(time.time()-t0)/60
                eta=elapsed*(refit_batches_target-seen_batches)/max(1,seen_batches)
                print(
                    f"refit batch={seen_batches}/{refit_batches_target} "
                    f"loss={running/seen_batches:.4f} elapsed={elapsed:.1f}m ETA={eta:.1f}m",
                    flush=True
                )

    # Use the same weight style that won heldout selection.
    selected_weight_style=str(best.get("weights_tag","ema"))
    use_refit_ema=(selected_weight_style=="ema") if REFIT_USE_SELECTED_WEIGHT_STYLE else True

    if use_refit_ema:
        print("Refit inference weights: EMA (matching heldout winner)")
        refit_context=refit_ema.applied()
    else:
        print("Refit inference weights: raw (matching heldout winner)")
        refit_context=contextlib.nullcontext()

    with refit_context:
        refit_model.eval()
        rtids,rtprob,_,rtatt=predict_loader(refit_model,test_loader)
        # Save the chosen refit model state while the desired weights are applied.
        REFIT_CKPT=ROOT/"refit_model_selected_weights.pt"
        atomic_torch_save({
            "model_state":{k:v.detach().cpu() for k,v in refit_model.state_dict().items()},
            "weights_tag":"ema" if use_refit_ema else "raw",
            "selected_passes":selected_passes,
            "thresholds_from_heldout":thresholds.astype(np.float32),
            "phase1_model_sha256":sha256_file(MODEL4_CKPT),
        },REFIT_CKPT)

    assert list(rtids)==list(tids),"Refit TEST order differs from heldout TEST order."

    # DO NOT tune thresholds on DEV after refit: refit trained on DEV.
    refit_thresholds=thresholds.copy()
    refit_pred_records=make_pred_records(
        rtids,rtprob,refit_thresholds,ensure_nonempty=True
    )

    REFIT_PRED=ROOT/"prediction_refit.json"
    json.dump(refit_pred_records,open(REFIT_PRED,"w",encoding="utf-8"),ensure_ascii=False)

    REFIT_ZIP=ROOT/"submission_exp1_REFIT.zip"
    with zipfile.ZipFile(REFIT_ZIP,"w",compression=zipfile.ZIP_DEFLATED) as z:
        z.write(REFIT_PRED,arcname="prediction.json")

    np.savez_compressed(
        ROOT/"refit_test_probs.npz",
        id=np.asarray(rtids),prob=rtprob,thresholds=refit_thresholds
    )

    print("REFIT mean predicted labels/document:",
          np.mean([len(r["pred"]) for r in refit_pred_records]))
    print("REFIT empty predictions:",
          sum(len(r["pred"])==0 for r in refit_pred_records))
    print("REFIT attention mean [HEAD,MIDDLE,EVIDENCE,TAIL]:",rtatt.mean(0))
    print("✅ REFIT submission:",REFIT_ZIP)
    REFIT_READY=True

    # Free optimizer state before ensemble / final validation.
    del refit_optimizer,refit_scaler,refit_backbone_params,refit_head_params
    gc.collect()
    if DEVICE.type=="cuda":
        torch.cuda.empty_cache()


REFIT: fresh Phase-1 Model4 -> TRAIN + DEV
TRAIN+DEV examples: 52931
selected heldout passes: 2.0
refit target batches: 52932
refit target optimizer updates: 6617
refit batch=250/52932 loss=0.7476 elapsed=0.4m ETA=94.6m
refit batch=500/52932 loss=0.6639 elapsed=0.9m ETA=92.3m
refit batch=750/52932 loss=0.5989 elapsed=1.3m ETA=90.8m
refit batch=1000/52932 loss=0.5595 elapsed=1.7m ETA=89.8m
refit batch=1250/52932 loss=0.5350 elapsed=2.2m ETA=89.2m
refit batch=1500/52932 loss=0.5181 elapsed=2.6m ETA=88.6m
refit batch=1750/52932 loss=0.5068 elapsed=3.0m ETA=88.1m
refit batch=2000/52932 loss=0.4968 elapsed=3.4m ETA=87.6m
refit batch=2250/52932 loss=0.4862 elapsed=3.9m ETA=87.1m
refit batch=2500/52932 loss=0.4765 elapsed=4.3m ETA=86.6m
refit batch=2750/52932 loss=0.4674 elapsed=4.7m ETA=86.2m
refit batch=3000/52932 loss=0.4582 elapsed=5.2m ETA=85.7m
refit batch=3250/52932 loss=0.4515 elapsed=5.6m ETA=85.3m
refit batch=3500/52932 loss=0.4450 elapsed=6.0m ETA=84.8m
refit batch=3750/52932 loss=

In [22]:
# CELL 15 — HELDOUT + REFIT probability ensemble
if not RUN_REFIT or not REFIT_READY:
    print("Refit unavailable — ensemble skipped.")
    ENSEMBLE_READY=False
else:
    assert abs(ENSEMBLE_HELDOUT_WEIGHT+ENSEMBLE_REFIT_WEIGHT-1.0)<1e-9

    # Same TEST IDs/order was asserted above.
    ensemble_prob=(
        ENSEMBLE_HELDOUT_WEIGHT*tprob +
        ENSEMBLE_REFIT_WEIGHT*rtprob
    ).astype(np.float32)

    # Reuse heldout-selected thresholds. We deliberately do not tune after refit,
    # because the refit model has already seen DEV labels.
    ensemble_thresholds=thresholds.copy()
    ensemble_pred_records=make_pred_records(
        tids,ensemble_prob,ensemble_thresholds,ensure_nonempty=True
    )

    ENSEMBLE_PRED=ROOT/"prediction_ensemble.json"
    json.dump(
        ensemble_pred_records,
        open(ENSEMBLE_PRED,"w",encoding="utf-8"),
        ensure_ascii=False
    )

    ENSEMBLE_ZIP=ROOT/"submission_exp1_ENSEMBLE.zip"
    with zipfile.ZipFile(
        ENSEMBLE_ZIP,"w",compression=zipfile.ZIP_DEFLATED
    ) as z:
        z.write(ENSEMBLE_PRED,arcname="prediction.json")

    np.savez_compressed(
        ROOT/"ensemble_test_probs.npz",
        id=np.asarray(tids),prob=ensemble_prob,thresholds=ensemble_thresholds,
        heldout_weight=np.float32(ENSEMBLE_HELDOUT_WEIGHT),
        refit_weight=np.float32(ENSEMBLE_REFIT_WEIGHT),
    )

    print("ENSEMBLE mean predicted labels/document:",
          np.mean([len(r["pred"]) for r in ensemble_pred_records]))
    print("ENSEMBLE empty predictions:",
          sum(len(r["pred"])==0 for r in ensemble_pred_records))
    print("✅ ENSEMBLE submission:",ENSEMBLE_ZIP)
    ENSEMBLE_READY=True


ENSEMBLE mean predicted labels/document: 4.845226207850065
ENSEMBLE empty predictions: 0
✅ ENSEMBLE submission: /kaggle/working/phase5_exp1_model4_partialft_v5/submission_exp1_ENSEMBLE.zip


In [23]:
# CELL 16 — validate ALL submission ZIPs with exact evaluate_phase5.py parser
import importlib.util
spec=importlib.util.spec_from_file_location("evaluate_phase5_exact",EVAL_SCRIPT)
eval_exact=importlib.util.module_from_spec(spec)
spec.loader.exec_module(eval_exact)

def validate_submission_zip(zip_path, expected_ids, n_labels=N_LABELS):
    zip_path=Path(zip_path)
    assert zip_path.exists(),zip_path
    check_dir=ROOT/("check_"+zip_path.stem)
    if check_dir.exists():
        shutil.rmtree(check_dir)
    check_dir.mkdir(parents=True,exist_ok=True)

    with zipfile.ZipFile(zip_path,"r") as z:
        names=z.namelist()
        assert names==["prediction.json"],f"{zip_path.name}: unexpected ZIP contents {names}"
        z.extract("prediction.json",check_dir)

    pred_path=check_dir/"prediction.json"
    pred=eval_exact.read_pred(str(pred_path),n_labels)
    assert set(pred)==set(expected_ids),(
        f"{zip_path.name}: ID mismatch: "
        f"missing={len(set(expected_ids)-set(pred))}, "
        f"extra={len(set(pred)-set(expected_ids))}"
    )
    print("✅",zip_path.name,
          "| rows:",len(pred),
          "| exact evaluator read_pred PASS")
    return True

EXPECTED_TEST_IDS={str(r["id"]) for r in test_records}

submission_zips=[
    ("HELDOUT",ROOT/"submission_exp1_HELDOUT.zip"),
]
if RUN_REFIT and REFIT_READY:
    submission_zips.append(("REFIT",ROOT/"submission_exp1_REFIT.zip"))
if RUN_REFIT and ENSEMBLE_READY:
    submission_zips.append(("ENSEMBLE",ROOT/"submission_exp1_ENSEMBLE.zip"))

for variant,zp in submission_zips:
    validate_submission_zip(zp,EXPECTED_TEST_IDS,N_LABELS)

assert len(submission_zips)==3 if RUN_REFIT else len(submission_zips)==1

print("\nREADY SUBMISSIONS")
for variant,zp in submission_zips:
    print(f"  {variant:9s} -> {zp}")

print("\nUpload these as separate Codabench submissions; their hidden-test ranking is not known locally.")


✅ submission_exp1_HELDOUT.zip | rows: 13019 | exact evaluator read_pred PASS
✅ submission_exp1_REFIT.zip | rows: 13019 | exact evaluator read_pred PASS
✅ submission_exp1_ENSEMBLE.zip | rows: 13019 | exact evaluator read_pred PASS

READY SUBMISSIONS
  HELDOUT   -> /kaggle/working/phase5_exp1_model4_partialft_v5/submission_exp1_HELDOUT.zip
  REFIT     -> /kaggle/working/phase5_exp1_model4_partialft_v5/submission_exp1_REFIT.zip
  ENSEMBLE  -> /kaggle/working/phase5_exp1_model4_partialft_v5/submission_exp1_ENSEMBLE.zip

Upload these as separate Codabench submissions; their hidden-test ranking is not known locally.


In [24]:
# CELL 15 — manifest / experiment record
manifest={
    "phase":5,
    "experiment":1,"version":"v5_heldout_refit_ensemble",
    "task":"Legal Statute Identification","notebook_version":NOTEBOOK_VERSION,
    "method":"Partial / Layer-wise Fine-tuning",
    "phase1_model":"Model4 Efficient Modern Causal Decoder",
    "architecture":{
        "vocab_size":VOCAB_SIZE,"d_model":D_MODEL,"n_layers":N_LAYERS,
        "n_heads":N_HEADS,"n_kv_heads":N_KV_HEADS,"ffn_dim":FFN_DIM,
        "max_positions":MAX_POSITIONS,"rope":True,"rmsnorm":True,"swiglu":True,
    },
    "adaptation":{
        "frozen_blocks":FROZEN_BLOCKS,
        "trainable_blocks":list(range(FROZEN_BLOCKS,N_LAYERS)),
        "n_chunks":N_CHUNKS,"chunk_len":CHUNK_LEN,
        "views":["head","middle","statute_evidence","tail"],
        "loss":"BCEWithLogits + sqrt(raw pos_weight), clipped [1,10]",
        "ema_decay":EMA_DECAY,"epochs":EPOCHS,"backbone_lr":BACKBONE_LR,"head_lr":HEAD_LR,
        "micro_batch":MICRO_BATCH_SIZE,"grad_accum":GRAD_ACCUM_STEPS,
    },
    "submission_variants":{
        "heldout":"best clean-DEV-selected checkpoint",
        "refit":"fresh Phase-1 initialization trained on TRAIN+DEV for heldout-selected duration",
        "ensemble":"0.5 heldout probability + 0.5 refit probability",
        "threshold_policy":"all variants use heldout-selected threshold vector; no post-refit DEV tuning"
    },
    "selection":{
        "primary":"cross-fitted clean DEV Macro-F1",
        "cv_macro_f1":best["cv_macro_f1"],"weights":best["weights_tag"],
        "best_epoch_frac":best["epoch_frac"],
        "threshold_strategy":best["threshold_strategy"],
        "clean_metrics":best["clean_metrics"],
        "official_metrics":best["official_metrics"],
    },
    "assets":{
        "model4_sha256":sha256_file(MODEL4_CKPT),
        "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
        "dataset_sha256":DATA_SHA256,
    },
    "data":{
        "train_rows":len(train_records),"official_dev_rows":len(dev_raw),
        "clean_dev_rows":len(clean_dev),"test_rows":len(test_records),
        "labels":N_LABELS,
    },
}
json.dump(manifest,open(ROOT/"manifest.json","w"),indent=2)
print(json.dumps(manifest,indent=2))
print("\nArtifacts in",ROOT)
for p in ROOT.iterdir():
    if p.is_file(): print(" ",p.name,round(p.stat().st_size/1e6,2),"MB")


{
  "phase": 5,
  "experiment": 1,
  "version": "v5_heldout_refit_ensemble",
  "task": "Legal Statute Identification",
  "notebook_version": "v5_HELDOUT_REFIT_ENSEMBLE",
  "method": "Partial / Layer-wise Fine-tuning",
  "phase1_model": "Model4 Efficient Modern Causal Decoder",
  "architecture": {
    "vocab_size": 24576,
    "d_model": 640,
    "n_layers": 17,
    "n_heads": 10,
    "n_kv_heads": 2,
    "ffn_dim": 2048,
    "max_positions": 2560,
    "rope": true,
    "rmsnorm": true,
    "swiglu": true
  },
  "adaptation": {
    "frozen_blocks": 12,
    "trainable_blocks": [
      12,
      13,
      14,
      15,
      16
    ],
    "n_chunks": 4,
    "chunk_len": 384,
    "views": [
      "head",
      "middle",
      "statute_evidence",
      "tail"
    ],
    "loss": "BCEWithLogits + sqrt(raw pos_weight), clipped [1,10]",
    "ema_decay": 0.999,
    "epochs": 3,
    "backbone_lr": 4e-05,
    "head_lr": 0.0004,
    "micro_batch": 2,
    "grad_accum": 8
  },
  "submission_variants":